# QQQ/TQQQ v4.1 churn diagnostics
Diagnostic-only analysis of state dwell, VXN-only exits and re-entry cycles.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from src.research.etf_rotation_experiment import fetch_adjusted_daily_bars
from src.research.vxn_attack_layer_long_history import run_attack_layer_comparison
from src.research.vxn_churn_diagnostics import state_dwell_table, round_trip_summary, reentry_cycles, transition_cost_by_reason, vxn_only_exit_events, summarize_churn


In [ ]:
diagnostic_path = Path('../configs/research_paradigms/qqq_tqqq_vxn_v4_1_churn_diagnostics.yaml')
diagnostic = yaml.safe_load(diagnostic_path.read_text(encoding='utf-8'))
base_path = Path('..') / diagnostic['base_contract']
base = yaml.safe_load(base_path.read_text(encoding='utf-8'))
bars, coverage = fetch_adjusted_daily_bars(symbols=['QQQ', 'TQQQ', '^VIX', '^VXN'], start=base['data']['start_date'])
metrics, results, prepared, _, tables = run_attack_layer_comparison(bars, base)


In [ ]:
baseline = results['attack_vix_v3_75']
overlay = results['attack_vxn_v4_1_75']
dwell = pd.concat([state_dwell_table(baseline), state_dwell_table(overlay)], ignore_index=True)
round_trip_summary(dwell, diagnostic['diagnostics']['round_trip_thresholds_sessions'])


In [ ]:
cycles = reentry_cycles(dwell, overlay.daily.index)
cycles


In [ ]:
events = vxn_only_exit_events(prepared, baseline, overlay, diagnostic['diagnostics']['event_horizons'])
events
